# Books Catalog Analysis with Python, Pandas and Web Scraping

This project collects book data from a scraping practice website and analyzes the catalog using Python and Pandas.

The goal is to demonstrate a complete data analysis workflow: web scraping, data cleaning, exploratory analysis, visualization and business interpretation.

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Style global des graphiques
plt.rcParams["font.family"] = ["Aptos Narrow", "Arial", "DejaVu Sans"]
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#D9D9D9"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#E6E6E6"
plt.rcParams["grid.linestyle"] = "-"
plt.rcParams["grid.linewidth"] = 0.7

In [54]:
# Palette portfolio
COLOR_BLUE = "#2F5D8C"
COLOR_TEAL = "#2FA4A9"
COLOR_LIGHT_BLUE = "#6FA3D2"
COLOR_GREEN = "#3BA272"
COLOR_YELLOW = "#F2C94C"
COLOR_ORANGE = "#F2994A"
COLOR_RED = "#D64541"
COLOR_PURPLE = "#9B51E0"

PALETTE = [
    COLOR_BLUE,
    COLOR_TEAL,
    COLOR_GREEN,
    COLOR_YELLOW,
    COLOR_ORANGE,
    COLOR_RED,
    COLOR_PURPLE,
    COLOR_LIGHT_BLUE
]

In [ ]:
df = pd.read_csv("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/books_raw.csv")
df.head()
df.shape
df.info

In [56]:
books = df.copy()

books["price"] = (
    books["price_raw"]
    .astype(str)
    .str.replace("£", "", regex=False)
    .str.replace("Â", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)
books.isnull().sum()
books["is_available"] = books["availability"].str.contains("In stock", case=False, na=False)
books_cleaned = books[["title", "price", "availability", "is_available", "rating_text", "rating"]]
books_cleaned.head()
books_cleaned.to_csv("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/books_cleaned.csv", index=False, encoding="utf-8")

In [57]:
def clean_chart(title=None, xlabel=None, ylabel=None):
    plt.title(title, fontweight="bold", pad=15)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    ax = plt.gca()

    # Supprimer les bordures inutiles
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Garder une grille légère seulement sur l'axe Y
    ax.grid(axis="y", alpha=0.35)
    ax.grid(axis="x", visible=False)

    plt.tight_layout()

In [ ]:
total_books = len(books_cleaned)
average_price = books_cleaned["price"].mean()
median_price = books_cleaned["price"].median()
average_rating = books_cleaned["rating"].mean()
available_rate = books_cleaned["is_available"].mean() * 100

summary = pd.DataFrame({
    "Metric": ["Total books", "Average price", "Median price", "Average rating", "Availability rate"],
    "Value": [total_books, round(average_price, 2), round(median_price, 2), round(average_rating, 2), round(available_rate, 2)]
})

summary

In [ ]:
plt.figure(figsize=(8, 5))

books_cleaned["price"].plot(
    kind="hist",
    bins=30,
    color=COLOR_BLUE,
    edgecolor="white"
)

clean_chart(
    title="Distribution des prix des livres",
    xlabel="Prix (£)",
    ylabel="Nombre de livres"
)

plt.savefig("screenshots/price_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
rating_distribution = books_cleaned["rating"].value_counts().sort_index()

plt.figure(figsize=(7, 5))

plt.bar(
    rating_distribution.index,
    rating_distribution.values,
    color=COLOR_TEAL
)

clean_chart(
    title="Répartition des notes",
    xlabel="Note",
    ylabel="Nombre de livres"
)

plt.xticks(rating_distribution.index)
plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/rating_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
availability_distribution = books_cleaned["is_available"].value_counts()

labels = ["Disponible" if x else "Indisponible" for x in availability_distribution.index]

plt.figure(figsize=(6, 4))

plt.bar(
    labels,
    availability_distribution.values,
    color=[COLOR_GREEN, COLOR_RED]
)

clean_chart(
    title="Disponibilité des livres",
    xlabel="Statut",
    ylabel="Nombre de livres"
)

plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/availability_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top_expensive_books = books_cleaned.sort_values("price", ascending=False).head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_expensive_books["title"],
    top_expensive_books["price"],
    color=COLOR_ORANGE
)

plt.gca().invert_yaxis()

clean_chart(
    title="Top 10 des livres les plus chers",
    xlabel="Prix (£)",
    ylabel="Titre du livre"
)

plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/top_expensive_books.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
price_by_rating = books_cleaned.groupby("rating")["price"].mean().reset_index()

plt.figure(figsize=(7, 5))

plt.bar(
    price_by_rating["rating"],
    price_by_rating["price"],
    color=COLOR_PURPLE
)

clean_chart(
    title="Prix moyen par note",
    xlabel="Note",
    ylabel="Prix moyen (£)"
)

plt.xticks(price_by_rating["rating"])
plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/price_by_rating.png", dpi=300, bbox_inches="tight")
plt.show()

## Key Insights

- The dataset contains 1,000 books scraped from a demo online catalog.
- Most books are available in stock.
- Prices are distributed across a wide range, allowing analysis of low and high-priced items.
- Ratings are spread from 1 to 5 stars.
- The most expensive books are not necessarily the highest-rated ones.
- Web scraping was used to collect the raw data before cleaning and analysis with Pandas.